# CONFIGURATION AND DATA LOAD/WRANGLING

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler       
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# --- Configuration ---
DATA_PATH = Path("./dataset") # <-- Change this to your dataset's root folder
SENSORS = ['P-MON-CKP', 'T-TPT', 'P-TPT'] # Select 3 key sensors
WINDOW_SIZE = 180 # seconds - 180 is a strong starting point for your later GRU model
STRIDE = 90 # seconds
RANDOM_SEED = 42

# --- Load and Combine Data ---
def load_data(path, sensors):
    all_data = []
    for folder in path.iterdir():
        if folder.is_dir() and folder.name.isdigit():
            label = int(folder.name)
            for file in folder.glob("*.parquet"):
                df = pd.read_parquet(file)
                # Select sensors and downsample to 1-minute intervals
                df = df[sensors].resample('1T').median()
                df.dropna(inplace=True)
                df['class'] = label
                all_data.append(df)
    return pd.concat(all_data, ignore_index=True)

df_full = load_data(DATA_PATH, SENSORS)
print(f"Total observations: {len(df_full)}")

Total observations: 973793


In [2]:
# --- Create Windows and Extract Features ---
def create_windows(df, window, stride, sensors):
    X, y = [], []
    for i in range(0, len(df) - window + 1, stride):
        window_data = df.iloc[i:i+window][sensors]
        # Calculate statistical features for each sensor in the window
        features = []
        for sensor in sensors:
            for stat in [np.mean, np.std, np.min, np.max, np.median]:
                features.append(stat(window_data[sensor]))
        X.append(features)
        y.append(df.iloc[i+window-1]['class']) # Label from last timestamp
    return np.array(X), np.array(y)

X_all, y_all = create_windows(df_full, WINDOW_SIZE, STRIDE, SENSORS)
print(f"Total windows created: {X_all.shape[0]}")
print(f"Features per window: {X_all.shape[1]}")

# --- Train/Test Split ---
# Separate features and labels, then split ensuring class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all
)

# Scale features using a StandardScaler, fitted only on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Prepare Normal Data for Training ---
# For one-class anomaly detection, train the model ONLY on 'Normal' class (label 0)
normal_X = X_train_scaled[y_train == 0]
print(f"Training on {normal_X.shape[0]} normal windows.")

Total windows created: 10818
Features per window: 15
Training on 971 normal windows.


In [3]:
# --- Train Baseline Isolation Forest Model ---
baseline_model = IsolationForest(
    n_estimators=200,
    contamination='auto',
    bootstrap=False,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
baseline_model.fit(normal_X)
print("Baseline model training complete.")

Baseline model training complete.


In [4]:
# --- Grid Search for Hyperparameter Tuning ---
from sklearn.model_selection import ParameterGrid

param_grid = {
    'n_estimators': [100, 200],
    'contamination': [0.01, 0.05, 0.1],
    'max_features': [0.5, 0.8, 1.0],
    'max_samples': [256, 512, 'auto']
}

best_f1, best_params = 0, {}
for params in ParameterGrid(param_grid):
    print(f"Testing: {params}")
    model = IsolationForest(**params, random_state=RANDOM_SEED, n_jobs=-1)
    model.fit(normal_X)
    
    # Predict and evaluate on test set
    preds = model.predict(X_test_scaled)
    # Convert (-1 = anomaly, 1 = normal) to (0 = normal, 1 = anomaly)
    y_pred = np.where(preds == 1, 0, 1)
    # Remap true labels: 0 (normal) -> 0, any other fault (1-9) -> 1
    y_true_bin = np.where(y_test == 0, 0, 1)
    
    f1 = f1_score(y_true_bin, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_params = params
        print(f"  *** New best F1: {f1:.4f} ***")

print(f"\nBest F1: {best_f1:.4f}")
print(f"Best hyperparameters: {best_params}")

Testing: {'contamination': 0.01, 'max_features': 0.5, 'max_samples': 256, 'n_estimators': 100}
  *** New best F1: 0.0124 ***
Testing: {'contamination': 0.01, 'max_features': 0.5, 'max_samples': 256, 'n_estimators': 200}
  *** New best F1: 0.0185 ***
Testing: {'contamination': 0.01, 'max_features': 0.5, 'max_samples': 512, 'n_estimators': 100}
  *** New best F1: 0.1007 ***
Testing: {'contamination': 0.01, 'max_features': 0.5, 'max_samples': 512, 'n_estimators': 200}
Testing: {'contamination': 0.01, 'max_features': 0.5, 'max_samples': 'auto', 'n_estimators': 100}
Testing: {'contamination': 0.01, 'max_features': 0.5, 'max_samples': 'auto', 'n_estimators': 200}
Testing: {'contamination': 0.01, 'max_features': 0.8, 'max_samples': 256, 'n_estimators': 100}
Testing: {'contamination': 0.01, 'max_features': 0.8, 'max_samples': 256, 'n_estimators': 200}
Testing: {'contamination': 0.01, 'max_features': 0.8, 'max_samples': 512, 'n_estimators': 100}
Testing: {'contamination': 0.01, 'max_features': 

In [5]:
# --- Train Final Optimised Model ---
final_model = IsolationForest(**best_params, random_state=RANDOM_SEED, n_jobs=-1)
final_model.fit(normal_X)
print("Final model training complete.")

Final model training complete.


In [6]:
# --- Evaluate and Save Model ---
preds = final_model.predict(X_test_scaled)
y_pred = np.where(preds == 1, 0, 1)
y_true_bin = np.where(y_test == 0, 0, 1)

print("\n--- Classification Report ---")
print(classification_report(y_true_bin, y_pred, target_names=['Normal', 'Anomaly']))
print(f"Binary Macro F1 Score: {f1_score(y_true_bin, y_pred, average='macro'):.4f}")


--- Classification Report ---
              precision    recall  f1-score   support

      Normal       0.97      0.91      0.94       243
     Anomaly       0.99      1.00      0.99      1921

    accuracy                           0.99      2164
   macro avg       0.98      0.95      0.97      2164
weighted avg       0.99      0.99      0.99      2164

Binary Macro F1 Score: 0.9666


In [7]:
# --- Save the final model and scaler for later use ---
model_filename = "final_isolation_forest_model.pkl"
joblib.dump(final_model, model_filename)
scaler_filename = "scaler.pkl"
joblib.dump(scaler, scaler_filename)
print(f"Model saved to {model_filename}, scaler saved to {scaler_filename}")

Model saved to final_isolation_forest_model.pkl, scaler saved to scaler.pkl


In [1]:
import json
import joblib
import numpy as np

# 1. Load your trained artifacts
scaler = joblib.load("scaler.pkl")
final_model = joblib.load("final_isolation_forest_model.pkl")

# 2. Extract Scaler properties
scaler_mean = scaler.mean_.tolist()
scaler_scale = scaler.scale_.tolist()

# 3. Recursive helper to parse scikit-learn's structural Cython Tree matrix
def parse_sklearn_tree(tree_obj, node_idx=0):
    left_child = int(tree_obj.children_left[node_idx])
    right_child = int(tree_obj.children_right[node_idx])
    
    if left_child == -1 and right_child == -1:
        return {
            "is_leaf": True,
            "size": int(tree_obj.n_node_samples[node_idx])
        }
    
    return {
        "is_leaf": False,
        "feature_idx": int(tree_obj.feature[node_idx]),
        "split_value": float(tree_obj.threshold[node_idx]),
        "left": parse_sklearn_tree(tree_obj, left_child),
        "right": parse_sklearn_tree(tree_obj, right_child)
    }

# 4. Serialize all 200 estimators along with their feature mappings
serialized_trees = []
for estimator, features in zip(final_model.estimators_, final_model.estimators_features_):
    tree_structure = parse_sklearn_tree(estimator.tree_)
    serialized_trees.append({
        "features": features.tolist(), 
        "tree": tree_structure
    })

# 5. Pack everything into a unified JSON object
model_payload = {
    "scaler_mean": scaler_mean,
    "scaler_scale": scaler_scale,
    "max_samples": int(final_model.max_samples_),
    "offset": float(final_model.offset_), 
    "trees": serialized_trees
}

# Save it to a file
with open("model_config.json", "w") as f:
    json.dump(model_payload, f, indent=2)

print("✅ Success! Your 'model_config.json' has been created.")


✅ Success! Your 'model_config.json' has been created.
